# Modeling and Backtesting

This notebook trains forecasting models using the feature artifacts generated in feature engineering and evaluates them with rolling-origin validation.

Goals:
1. Benchmark against strong baselines.
2. Use fixed temporal splits from feature engineering.
3. Compare models with MAE as primary metric and RMSE/MAPE as guardrails.
4. Report performance by fold and by temporal segments.

In [ ]:
# Imports and project setup.
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# Load processed artifacts from feature engineering.
processed_dir = project_root / "data" / "processed"
features_parquet = processed_dir / "features.parquet"
features_csv = processed_dir / "features.csv"
split_csv = processed_dir / "split_plan.csv"

if features_parquet.exists():
    df = pd.read_parquet(features_parquet)
elif features_csv.exists():
    df = pd.read_csv(features_csv, index_col=0, parse_dates=True)
else:
    raise FileNotFoundError(
        "No features artifact found. Run feature_engineering notebook first to generate features.parquet or features.csv"
    )

split_plan = pd.read_csv(split_csv, parse_dates=["train_start", "train_end", "valid_start", "valid_end"])

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

print(f"Feature dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} -> {df.index.max()}")
display(split_plan)

In [ ]:
# Modeling contract.
TARGET_COL = "Global_active_power"
PRIMARY_METRIC = "MAE"
GUARDRAIL_METRICS = ["RMSE", "MAPE"]

if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not present in feature dataset.")

feature_cols = [c for c in df.columns if c != TARGET_COL]
print("Number of features:", len(feature_cols))
print("Sample features:", feature_cols[:10])

In [ ]:
# Metrics and helpers.
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    safe = np.where(denom == 0, 1.0, denom)
    return np.mean(np.abs(y_true - y_pred) / safe) * 100


def metric_bundle(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100),
        "sMAPE": float(smape(y_true, y_pred)),
    }


def evaluate_by_segment(index, y_true, y_pred):
    seg = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "hour": index.hour,
        "day_of_week": index.dayofweek,
        "month": index.month,
    }, index=index)

    rows = []
    for segment_name, group_col in [("hour", "hour"), ("day_of_week", "day_of_week"), ("month", "month")]:
        for k, g in seg.groupby(group_col):
            m = metric_bundle(g["y_true"], g["y_pred"])
            rows.append({
                "segment": segment_name,
                "segment_value": int(k),
                "MAE": m["MAE"],
                "RMSE": m["RMSE"],
                "MAPE": m["MAPE"],
                "sMAPE": m["sMAPE"],
                "n": len(g),
            })
    return pd.DataFrame(rows)

## Rolling Backtest

Models:
1. Naive lag-1 baseline.
2. Naive lag-24 baseline.
3. RandomForest regressor as a non-linear benchmark.

In [ ]:
# Rolling-origin evaluation loop.
all_fold_results = []
all_segment_results = []
all_predictions = []

for _, row in split_plan.iterrows():
    fold = int(row["fold"])

    train_mask = (df.index >= row["train_start"]) & (df.index <= row["train_end"])
    valid_mask = (df.index >= row["valid_start"]) & (df.index <= row["valid_end"])

    train_df = df.loc[train_mask]
    valid_df = df.loc[valid_mask]

    X_train = train_df[feature_cols]
    y_train = train_df[TARGET_COL]
    X_valid = valid_df[feature_cols]
    y_valid = valid_df[TARGET_COL]

    if len(train_df) == 0 or len(valid_df) == 0:
        print(f"Skipping fold {fold}: empty train or validation set")
        continue

    preds = {}

    # Baseline 1: persistence (lag-1)
    lag1_col = f"{TARGET_COL}_lag_1"
    if lag1_col in X_valid.columns:
        preds["baseline_lag1"] = X_valid[lag1_col].values

    # Baseline 2: daily seasonal persistence (lag-24)
    lag24_col = f"{TARGET_COL}_lag_24"
    if lag24_col in X_valid.columns:
        preds["baseline_lag24"] = X_valid[lag24_col].values

    # Tree model benchmark
    model = RandomForestRegressor(
        n_estimators=250,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    preds["random_forest"] = model.predict(X_valid)

    for model_name, y_pred in preds.items():
        m = metric_bundle(y_valid.values, y_pred)
        all_fold_results.append({
            "fold": fold,
            "model": model_name,
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "MAPE": m["MAPE"],
            "sMAPE": m["sMAPE"],
            "n": len(y_valid),
        })

        seg_res = evaluate_by_segment(valid_df.index, y_valid.values, y_pred)
        seg_res["fold"] = fold
        seg_res["model"] = model_name
        all_segment_results.append(seg_res)

        pred_df = pd.DataFrame({
            "timestamp": valid_df.index,
            "fold": fold,
            "model": model_name,
            "y_true": y_valid.values,
            "y_pred": y_pred,
        })
        all_predictions.append(pred_df)

fold_results = pd.DataFrame(all_fold_results)
segment_results = pd.concat(all_segment_results, ignore_index=True) if all_segment_results else pd.DataFrame()
predictions = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()

display(fold_results.sort_values(["model", "fold"]))

In [ ]:
# Leaderboard and stability view.
if fold_results.empty:
    raise RuntimeError("No fold results produced. Check split windows and feature artifacts.")

leaderboard = (
    fold_results.groupby("model", as_index=False)[["MAE", "RMSE", "MAPE", "sMAPE"]]
    .mean()
    .sort_values("MAE")
)

display(leaderboard)

stability = (
    fold_results.groupby("model", as_index=False)[["MAE", "RMSE", "MAPE", "sMAPE"]]
    .std()
    .rename(columns={"MAE": "MAE_std", "RMSE": "RMSE_std", "MAPE": "MAPE_std", "sMAPE": "sMAPE_std"})
)

display(stability)

In [ ]:
# Segment diagnostics: where each model underperforms.
segment_summary = (
    segment_results.groupby(["model", "segment", "segment_value"], as_index=False)[["MAE", "RMSE", "MAPE", "sMAPE", "n"]]
    .mean()
)

# Worst 10 segments by MAE per model.
for model_name in segment_summary["model"].unique():
    print(f"\nWorst segments for {model_name} by MAE")
    display(segment_summary[segment_summary["model"] == model_name].sort_values("MAE", ascending=False).head(10))

In [ ]:
# Persist evaluation artifacts.
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

fold_results_path = reports_dir / "model_fold_results.csv"
leaderboard_path = reports_dir / "model_leaderboard.csv"
segment_path = reports_dir / "model_segment_results.csv"
preds_path = reports_dir / "model_predictions.csv"

fold_results.to_csv(fold_results_path, index=False)
leaderboard.to_csv(leaderboard_path, index=False)
segment_summary.to_csv(segment_path, index=False)
predictions.to_csv(preds_path, index=False)

print("Saved:")
print(fold_results_path)
print(leaderboard_path)
print(segment_path)
print(preds_path)